In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)  # keeps results reproducible every time you re-run this

n = 372

df = pd.DataFrame({
    'RespondentID': range(1, n + 1),
    'Gender': np.random.choice(['Male', 'Female'], n),
    'Age': np.random.choice(range(18, 80), n),
    'Income': np.random.choice(
        ['Under $30,000', '$30,000-$49,999', '$50,000-$74,000',
         '$75,000-$99,999', '$100,000 or over'],
        n, p=[0.14, 0.26, 0.28, 0.13, 0.19]
    ),
    'Education': np.random.choice(
        ['Some High School or less', 'High School Graduate',
         'Some College/Technical School', 'College Graduate or higher'],
        n, p=[0.02, 0.09, 0.31, 0.58]
    ),
    'HoursOnline': np.random.choice(
        ['Less than 1 hour', '1 to 5 hours', '6 to 10 hours',
         '11 to 20 hours', '21 to 40 hours', '41 hours or more'],
        n, p=[0.054, 0.374, 0.282, 0.148, 0.091, 0.051]
    ),
    'Satisfaction': np.random.choice(
        ['Very satisfied', 'Somewhat satisfied',
         'Somewhat dissatisfied', 'Very dissatisfied'],
        n, p=[0.559, 0.395, 0.043, 0.003]
    ),
    'PurchaseIntent_10pctHigher': np.random.choice(
        ['Definitely would purchase', 'Probably would purchase',
         'Might or might not have purchased', 'Probably would not purchase',
         'Definitely would not purchase'],
        n, p=[0.03, 0.13, 0.20, 0.31, 0.33]
    ),
})

assert df.isnull().sum().sum() == 0, "Missing values found!"
print("N Valid =", len(df), "| Missing =", df.isnull().sum().sum())

df.to_csv('dell_survey.csv', index=False)
print(df.head())

N Valid = 372 | Missing = 0
   RespondentID  Gender  Age            Income                      Education  \
0             1    Male   43   $75,000-$99,999  Some College/Technical School   
1             2  Female   65   $50,000-$74,000  Some College/Technical School   
2             3    Male   74  $100,000 or over     College Graduate or higher   
3             4    Male   69   $75,000-$99,999     College Graduate or higher   
4             5    Male   77   $75,000-$99,999     College Graduate or higher   

     HoursOnline        Satisfaction         PurchaseIntent_10pctHigher  
0  6 to 10 hours      Very satisfied      Definitely would not purchase  
1   1 to 5 hours      Very satisfied      Definitely would not purchase  
2   1 to 5 hours  Somewhat satisfied  Might or might not have purchased  
3   1 to 5 hours      Very satisfied        Probably would not purchase  
4   1 to 5 hours      Very satisfied      Definitely would not purchase  


In [2]:
import pandas as pd

df = pd.read_csv('dell_survey.csv')

# Convert Gender to 0/1
df['Gender_num'] = df['Gender'].map({'Male': 0, 'Female': 1})

# Convert Income brackets to an ordered scale 1-5 (low to high)
income_map = {
    'Under $30,000': 1,
    '$30,000-$49,999': 2,
    '$50,000-$74,000': 3,
    '$75,000-$99,999': 4,
    '$100,000 or over': 5
}
df['Income_num'] = df['Income'].map(income_map)

# Convert Purchase Intention to an ordered scale 1-5 (low intent to high intent)
intent_map = {
    'Definitely would not purchase': 1,
    'Probably would not purchase': 2,
    'Might or might not have purchased': 3,
    'Probably would purchase': 4,
    'Definitely would purchase': 5
}
df['Intent_num'] = df['PurchaseIntent_10pctHigher'].map(intent_map)

print(df[['Gender', 'Gender_num', 'Income', 'Income_num', 'PurchaseIntent_10pctHigher', 'Intent_num']].head())

   Gender  Gender_num            Income  Income_num  \
0    Male           0   $75,000-$99,999           4   
1  Female           1   $50,000-$74,000           3   
2    Male           0  $100,000 or over           5   
3    Male           0   $75,000-$99,999           4   
4    Male           0   $75,000-$99,999           4   

          PurchaseIntent_10pctHigher  Intent_num  
0      Definitely would not purchase           1  
1      Definitely would not purchase           1  
2  Might or might not have purchased           3  
3        Probably would not purchase           2  
4      Definitely would not purchase           1  


In [3]:
import statsmodels.api as sm

X = df[['Gender_num', 'Income_num']]
X = sm.add_constant(X)
y = df['Intent_num']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             Intent_num   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.003
Method:                 Least Squares   F-statistic:                    0.5358
Date:                Tue, 28 Jul 2026   Prob (F-statistic):              0.586
Time:                        15:15:10   Log-Likelihood:                -568.44
No. Observations:                 372   AIC:                             1143.
Df Residuals:                     369   BIC:                             1155.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.1474      0.148     14.500      0.0

"A multiple regression testing whether Gender and Income predict purchase intention (at a 10% price increase) found no statistically significant relationship (F-statistic p=0.586, R²=0.3%). Neither Gender (p=0.427) nor Income (p=0.470) individually predicted purchase intent. This suggests demographic factors alone do not explain customers' price sensitivity — other factors (e.g., brand loyalty, perceived product value, or competitor pricing) likely play a larger role and warrant further investigation."